# 다음 순서 1번 — 9B에서 J-Lens가 logit lens보다 나은가

> 9B에서 105문항으로 J-Lens를 맞추고 별도 45문항의 레이어·토큰 신호를 확인한다.
> 일반 logit lens와 비교하되, 신호만으로 원인을 주장하지 않는다.
> — 2026-10-03 「다음 순서」 1

10/2에 2B로 같은 비교를 해서 **0.35 vs 0.10**이 나왔지만, 차이의 신뢰구간에 0이 들어가 우위를 확정하지 못했다.
현상이 가장 깨끗한 9B에서 다시 재는 것이 이 노트북의 전부다.

**이 노트북은 운전석이다.** 계산 로직은 전부 `mindscopex_analysis.jlens`와
`experiments/jobs/jlens_item1.py`에 있고 단위 테스트가 붙어 있다. 여기서는 부르기만 한다.

## 읽는 법

| 숫자 | 뜻 |
|---|---|
| 상관(correlation) | 그 층의 렌즈 읽기가 **모델의 최종 선택을 얼마나 미리 알려주나**. 1에 가까울수록 잘 맞춤 |
| delta | `J-Lens − logit lens`. 양수면 J-Lens가 나음 |
| CI | 문항을 다시 뽑아가며(bootstrap) 잰 delta의 범위. **0을 포함하면 "확정 못 함"** |

## 하지 않는 것

읽기(readout)는 인과 증거가 아니다. 어떤 층의 신호가 최종 답을 잘 예측한다고 해서
그 층이 답을 **만든다**는 뜻이 아니다. 논문도 둘을 분리해서 다룬다. 인과 확인은 3번 작업이다.

## 1. 환경

Colab에서만 설치가 돈다. 로컬에서는 건너뛴다.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO = "/content/mindscopex_analysis" if IN_COLAB else os.path.abspath("..")

if IN_COLAB:
    if not os.path.exists(REPO):
        # 포크 주소로 바꿔서 쓸 것
        !git clone https://github.com/potter0923/mindscopex_analysis.git {REPO}
    !pip -q install -e {REPO}
    !pip -q install git+https://github.com/anthropics/jacobian-lens.git

sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, REPO)
print("colab:", IN_COLAB, "| repo:", REPO)

In [ ]:
# GPU 확인. 9B는 가중치만 18GB라 무료 T4(16GB)에는 들어가지 않는다.
try:
    import torch

    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"{name}  {gb:.0f} GB")
        print("9B 가능" if gb >= 24 else "9B 불가 — PROFILE='2b'로 리허설만")
    else:
        print("GPU 없음 — 아래 2절 dry-run까지만 돈다")
except ImportError:
    print("torch 없음 — dry-run까지만")

## 2. 계획 확인 (dry-run)

모델을 올리기 **전에** 분할·층·비용을 먼저 본다. GPU를 켜둔 채 오타를 잡으면 시간이 녹는다.

In [ ]:
from experiments.jobs import jlens_item1

PROFILE = "9b"  # 무료 T4뿐이면 "2b"
N_FIT = 105  # 처음에는 20으로 파이프라인부터 확인
TAG = "main"

args = jlens_item1.parse_args(
    ["--profile", PROFILE, "--n-fit", str(N_FIT), "--tag", TAG, "--dry-run"]
)
plan = jlens_item1.build_plan(args)
print(jlens_item1.describe(plan))

In [ ]:
# 적합용과 측정용이 정말 안 겹치는지 눈으로 확인
fit_ids = {c.case_id for c in plan["fit_cases"]}
eval_ids = {c.case_id for c in plan["eval_cases"]}
print("겹침:", fit_ids & eval_ids or "없음")
print("측정 문항 예시:", [c.case_id for c in plan["eval_cases"][:3]])
print()
print(plan["eval_cases"][0].prompt)
print("정답:", repr(plan["eval_cases"][0].correct_answer))
print("함정:", repr(plan["eval_cases"][0].lure_answer))

### 2-1. 답이 토큰으로 구분되는지 — 반드시 먼저 확인

렌즈는 **한 위치의 한 토큰**을 읽는다. 그런데 `hagendorff_crt`의 답은 `" $20.0"` / `" $40.0"` 꼴이라
**첫 토큰이 둘 다 `" $"`로 같다.** 첫 토큰만 보면 150문항 전부가 측정 불가가 된다.

`answer_contrast()`는 **두 답이 처음으로 갈라지는 위치**를 찾아서, 공통 앞부분(`" $"`)을
프롬프트에 붙인 뒤 그 다음 자리를 읽는다. 아래 셀로 실제 토크나이저에서 확인한다.

In [ ]:
from collections import Counter

from mindscopex_analysis import jlens
from mindscopex_analysis.models import load_qwen_text_generation_model

hf_model, tokenizer = load_qwen_text_generation_model(plan["model_id"])

prefix_counts, unmeasurable = Counter(), []
for case in plan["eval_cases"]:
    try:
        prefix_counts[jlens.answer_contrast(tokenizer, case).prefix_tokens] += 1
    except ValueError:
        unmeasurable.append(case.case_id)

print("붙여야 하는 prefix 토큰 수 분포:", dict(sorted(prefix_counts.items())))
print("측정 불가 문항:", unmeasurable or "없음")

sample = jlens.answer_contrast(tokenizer, plan["eval_cases"][0])
print("\n예시 — prefix:", repr(sample.prefix))
print("읽을 토큰:", tokenizer.decode([sample.correct_token_id]), "vs", tokenizer.decode([sample.lure_token_id]))

## 3. 실행

렌즈 적합이 오래 걸린다. 9B·105문항은 **문항당 역전파 256회 × 105 = 26,880회**라 몇 시간이다.

- 노트북이 끊겨도 `lens.pt` / `lens_checkpoint.pt`가 남아 **이어서 간다**
- 몇 시간짜리는 노트북보다 터미널에서 돌리는 쪽이 안전하다:

```bash
python -m experiments.jobs.jlens_item1 --profile 9b --n-fit 105 --tag main
```

In [ ]:
run_args = jlens_item1.parse_args(
    ["--profile", PROFILE, "--n-fit", str(N_FIT), "--tag", TAG]
)
result = jlens_item1.run(run_args)

## 4. 결과

`delta`의 신뢰구간이 **0을 넘지 못하면** 10/2와 같은 상태다 — 더 좋아 *보이지만* 확정은 못 한다.
그 경우에도 결과다. "9B에서도 확정 불가"는 보고할 가치가 있다.

In [ ]:
import pandas as pd

table = pd.DataFrame(result["correlations"]).set_index("layer")
display(table.round(3))

ci = result["delta_ci"]
if ci:
    verdict = "J-Lens 우위 확정" if ci["low"] > 0 else "확정 불가 (구간에 0 포함)"
    print(f"최고 층 {ci['layer']}: delta {ci['delta']:+.3f} [{ci['low']:+.3f}, {ci['high']:+.3f}] → {verdict}")

In [ ]:
ax = table[["jacobian", "logit"]].plot(figsize=(9, 4), marker="o")
ax.axhline(0, color="0.7", linewidth=0.8)
ax.set_xlabel("layer")
ax.set_ylabel("최종 선택과의 상관 (Spearman)")
ax.set_title(f"{result['config']['model_id']} · {result['config']['dataset']} · n_eval={result['config']['n_eval']}")
ax.legend(["J-Lens", "logit lens"])

## 5. 한계

- **읽기 ≠ 인과.** 상관이 높은 층이 답을 결정한다는 뜻이 아니다. 개입 실험은 3번 작업이다.
- **단일 토큰.** 두 답이 갈라지는 한 위치만 본다. 긴 서술형 답에는 그대로 못 쓴다.
- **앞쪽 층.** 논문은 모델 앞 1/3의 읽기를 "noisy and largely uninterpretable"이라고 명시한다.
- **층 간격.** 비용 때문에 2층 간격으로 훑는다. 좁은 밴드는 후보가 좁혀진 뒤 다시 본다.
- **데이터.** `hagendorff_crt`는 2023년 공개 데이터다. 같은 설정을 `crt_fresh_v2`로도 돌려보면
  `--dataset crt_fresh_v2` 한 줄로 비교할 수 있다 (2번 작업과 이어지는 부분).